![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 01 | Exploratory Data Analysis

*Project 1 — SQL: From Data to Insight*

**Team:**
**Dataset:**

---

### What this notebook is for

Understanding your data before you design anything. You cannot decide which tables you need until you know which columns are categorical, which are broken, and which are identifiers. That is the whole job here.

Two of these sections feed directly into tomorrow: **6. Categorical cardinality** finds your lookup tables, and **7. Research questions** decides which columns you actually need to keep.

> **Done when:** you can name your 3+ tables, you have written down at least 2 research questions, and you know every data-quality problem you will have to fix in notebook 02.

### How to work in here

Each section has a starter call. Run it, read the output, then **write down what you see in the markdown cell below it** — not for marks, but because notebook 03 asks you to justify decisions you made today and you will not remember them.

Functions live in [`../src/functions.py`](../src/functions.py). Some are written for you; some are a `TODO` with a docstring explaining the decisions to make. Add your own as you go.

---
## 0. Setup

In [ ]:
import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.functions import *

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")

---
## 1. Load the raw data

`load_raw` reads from `data/raw/`, and handles `.csv`, `.csv.gz` and `.xlsx` so you do not have to remember which reader to use.

Run `python download_data.py <payments|airbnb|retail>` from the repo root first.

In [ ]:
# Change this to your dataset's file. Options after download_data.py:
#   payments -> "cash_requests.csv", "fees.csv"
#   airbnb   -> "listings.csv.gz", "reviews.csv.gz"
#   retail   -> "online_retail_II.csv"
df = load_raw("cash_requests.csv")

df.shape

In [ ]:
# Working with two files (payments, or airbnb)? Load the second one too.
# df2 = load_raw("fees.csv")
# df2.shape

---
## 2. First look

Three questions: how big is it, what type is every column, and what does a row look like.

`overview` answers the first two in one table — dtype, nulls, % null, distinct values and a real example per column, sorted so the emptiest columns come first.

In [ ]:
overview(df)

In [ ]:
df.head()

In [ ]:
df.describe()          # numeric columns
# df.describe(include="object")   # and the text ones

**What you see:**

- Rows and columns:
- Columns that are obviously identifiers (one distinct value per row):
- Columns whose dtype is wrong (dates read as text, numbers read as text):
- Columns you will not need at all:

---
## 3. Missing values

`missing_report` lists only the columns that have any, worst first. Pass a threshold to see just the badly affected ones.

For each one you need a decision, and "drop the column" is a legitimate one — a column that is 95% empty cannot support an analysis.

In [ ]:
missing_report(df)

In [ ]:
missing_report(df, threshold=0.5)      # more than half empty

**Decisions** (carry these into notebook 02):

| Column | % missing | What it means when it is missing | Decision |
|---|---|---|---|
|  |  |  |  |

A missing value usually *means* something. In the Payments data an empty `money_back_date` means the advance was never repaid, which is a finding, not a gap to fill.

---
## 4. Duplicates and blank strings

Two separate problems that look the same.

**Duplicates** are rows that repeat. Pass `subset=` to check a candidate primary key: if `subset=["id"]` reports any duplicates, `id` cannot be your primary key.

**Blank strings** are `""` and `" "`. They are not `NaN`, so `isna()` does not see them and `dropna()` does not remove them — they sail through into your database as an empty-string category.

In [ ]:
duplicate_report(df)

In [ ]:
# Is your candidate primary key actually unique?
duplicate_report(df, subset=["id"])

**What you see:**

- Fully duplicated rows:
- Is your primary key unique?
- Columns with blank strings to clean:

---
## 5. Numerical distributions

Look at the numbers you plan to analyse. You are looking for three things: the shape (symmetric or skewed), outliers, and values that are impossible.

An impossible value is the one that matters most — a negative price, an age of 300, a review date before the platform existed. Those are data errors and they will distort every average you compute on Wednesday.

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
numeric_cols

In [ ]:
# Pick the 3-4 numeric columns your research questions depend on.
cols = numeric_cols[:4]

fig, axes = plt.subplots(1, len(cols), figsize=(4 * len(cols), 3))
for ax, col in zip(np.atleast_1d(axes), cols):
    df[col].plot(kind="hist", bins=40, ax=ax, title=col)
plt.tight_layout()

In [ ]:
# Boxplots make the outliers obvious.
df[cols].plot(kind="box", subplots=True, layout=(1, len(cols)), figsize=(4 * len(cols), 3))
plt.tight_layout()

**What you see:**

- Skewed columns (mean far from median):
- Outliers — real extremes, or errors?
- Impossible values:

---
## 6. Categorical cardinality — **this is the important one**

This section decides your database design.

`categorical_report` lists every text column with few distinct values. Each one is a candidate lookup table: that is Option A in the [brief](../README.md#option-a--split-one-dataset-into-3-tables-recommended).

How to read the output:

- **2 to 30 distinct values** → a lookup table. `status` with 7 values across 24,000 rows is the textbook case.
- **Hundreds of distinct values, each repeating** → a dimension table, not a lookup. A host, a customer, a product.
- **As many distinct values as rows** → an identifier. Leave it in the fact table.

In [ ]:
categorical_report(df)

In [ ]:
# Raise the ceiling if your categories are genuinely numerous - neighbourhood,
# country, product category.
categorical_report(df, max_unique=100)

In [ ]:
# Look at the actual distribution before you commit. A "category" where 99.8%
# of rows share one value will not tell you anything.
df["status"].value_counts(dropna=False)

**Your lookup-table candidates:**

| Column | Distinct values | Becomes table | Why it is worth splitting out |
|---|---|---|---|
|  |  |  |  |

Pick **three or four**, not everything on the list. A schema of eight two-row tables is worse design than three meaningful ones, and the rubric grades whether your relationships make sense — not how many you have.

---
## 7. Research questions

At least two, and they are what everything downstream is judged against. Notebook 03 restates them, your queries answer them, and your conclusions slide says whether they held.

A usable question is **specific**, **answerable with this data**, and has a **plausible answer you could be wrong about**:

- Good: *"Do instant transfers have a higher incident rate than regular ones?"* — one number each, and it settles the question.
- Too vague: *"What are the trends in the data?"*
- Not answerable here: *"Would users pay more for faster transfers?"* — nothing in the data speaks to willingness to pay.

For each question write the **business hypothesis**: what you expect, and why. Being wrong is a fine outcome; having no prior is not.

> Business hypothesis, not statistical. "Entire homes are priced higher per night than private rooms" is what we want. Significance testing with p-values is Week 4 — you are not expected to run a t-test and you are not graded on one.

### Q1:
**Hypothesis:**
**Columns it needs:**

### Q2:
**Hypothesis:**
**Columns it needs:**

### Q3 *(optional)*:
**Hypothesis:**
**Columns it needs:**

### Who cares about the answer?

One paragraph: whose decision would change if you are right? That paragraph is your slide 2.

---
## 8. Data-quality summary

Close the notebook with the to-do list for tomorrow. Everything you noticed above, in one place, so notebook 02 is execution rather than rediscovery.

| # | Problem | Columns affected | Fix in notebook 02 |
|---|---|---|---|
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |

### Planned tables

| Table | Type | Primary key | Foreign keys | Approx. rows |
|---|---|---|---|---|
|  | lookup / dimension / fact |  |  |  |

### Where you are

- [ ] I can name my 3+ tables and their keys
- [ ] I have at least 2 research questions written down
- [ ] I know every cleaning step I have to do
- [ ] Committed and pushed

Next: **[02_processing.ipynb](02_processing.ipynb)** — clean it, design it, load it.